# B：接收输出复制到普通 CPU 数组
同一缓存排版，比较direct与snapshot；一次原B加载、每组3预热+30计时，仍逐字核验/哈希/原生显示。CPU复制成本单独计入，不修改官方驱动、冻结算法或原生帧缓存。红色报错或一分钟无进展时保留输出并停止，不重复运行、不释放缓冲、不关闭失败内核或切换Overlay。

## 1. 准备函数
In [数字]且无错误表示完成，无打印正常。

In [ ]:
from b_snapshot_board_test import prepare_snapshot, load_snapshot, start_snapshot, check_snapshot, compare_snapshot, finish_snapshot


## 2. 检查文件与普通 CPU 暂存
应出现 B_SNAPSHOT_PRECHECK_PASS；本框不加载FPGA。

In [ ]:
if 'snapshot_context' in globals():
    raise RuntimeError('Already prepared; do not rerun')
snapshot_retained = []
snapshot_context = prepare_snapshot('.', snapshot_retained)


## 3. 加载一次
应出现 B_SNAPSHOT_LOAD_PASS，原B只加载一次。

In [ ]:
load_snapshot(snapshot_context)


## 4. 短帧与 HDMI 准备
应出现 B_SNAPSHOT_HDMI_READY；彩条蓝/绿/红与白色定位线正常再继续。

In [ ]:
start_snapshot(snapshot_context)


## 5. 实际 DMA/CPU 复制与整幅像素门禁
应出现 B_SNAPSHOT_PIXELS_PASS。核对262,144个真实输出字、4个标记的整幅图像，再进入计时。

In [ ]:
check_snapshot(snapshot_context)


## 6. 两组对比
先direct再snapshot，每组3预热+30计时。snapshot复制后仍完整逐字核验、排版和哈希；结束 B_SNAPSHOT_COMPARE_COMPLETE。复制成本计入完整循环，不从5.016旧结果推断本次提速。

In [ ]:
compare_snapshot(snapshot_context)
for group in snapshot_context['report']['snapshot_compare']['groups']:
    summary = group['summary']
    print('{}: checked loop {:.6f} FPS'.format(group['name'], summary['loop_fps']))
    for name, stage in summary['stages'].items():
        print('  {}: {:.6f} ms, {:.3f}%'.format(name, stage['mean_ms'], stage['share_of_sample_time_percent']))
print('CPU copy cost included; independent verification pending.')


## 7. 关闭并打包
应出现 B_SNAPSHOT_RESULTS_READY 和最终预览；显示器无信号正常。Ctrl+S，File → Download as → Notebook下载执行记录，回到列表下载 b_snapshot_bundle.zip。发布NOT_RUN/原JSON视觉PENDING保留，待独立核验。

In [ ]:
finish_snapshot(snapshot_context)
from IPython.display import Image, display
display(Image(filename=str(snapshot_context['directory'] / 'b_snapshot_snapshot_final.png')))
